# Create results table
Creates the central table that stores every validation run's results, plus a view of the latest result per check.

In [0]:
%sql
-- Keep results separate from test data and configuration
create schema if not exists validation_sweep.results;

-- Central results table for all validation runs
create table if not exists validation_sweep.results.validation_results (
  run_id              string        not null comment 'Identifies one validation run; shared by all rows written in that run',
  run_ts              timestamp     not null comment 'When the run started (UTC)',
  run_date            date          not null comment 'Date of the run (UTC)',
  check_id            string        not null comment 'Identifies a check by method, test_type, table_name, and key_columns',
  method              string        not null comment 'Validation approach, for example python_native',
  test_type           string        not null comment 'Kind of check, for example pk_integrity',
  table_name          string        not null comment 'Full name of the checked table: catalog.schema.table',
  key_columns         array<string>          comment 'Primary key columns checked, from the registry',
  test_result         string        not null comment 'PASS, FAIL or ERROR',
  row_count           bigint                 comment 'Total rows in the table',
  distinct_key_count  bigint                 comment 'Distinct key values, excluding rows with a null key column',
  null_key_count      bigint                 comment 'Rows where any key column is null',
  duplicate_count     bigint                 comment 'Extra rows sharing a non-null key value',
  error_message       string                 comment 'First line of the error when test_result is ERROR',
  constraint validation_results_pk primary key (run_id, check_id)
)
cluster by (run_date, table_name)
comment 'Append-only history of validation results; one row per check per run. Unique key: run_id + check_id';

-- Results for all checks from the most recent run of each method and test type.
-- Matching on run_id keeps a run's rows together even if their timestamps differ.
create or replace view validation_sweep.results.latest_validation_results as
select *
from validation_sweep.results.validation_results
qualify run_id = first_value(run_id) over (
  partition by method, test_type
  order by run_ts desc, run_id desc
);

In [0]:
# Verify the results table has the expected columns
expected_columns = [
    "run_id", "run_ts", "run_date", "check_id", "method", "test_type", "table_name", "key_columns",
    "test_result", "row_count", "distinct_key_count", "null_key_count", "duplicate_count", "error_message",
]

actual_columns = spark.table("validation_sweep.results.validation_results").columns

if actual_columns != expected_columns:
    raise ValueError(f"Unexpected columns in results table:\n"
                     f"expected {expected_columns}\n"
                     f"found    {actual_columns}")
      
# Verify the latest-results view can be queried, even when empty
spark.table("validation_sweep.results.latest_validation_results").limit(1).collect()
print("Results table and latest-results view are ready")